## REINFORCE and Lunar Lander

In [ ]:
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import image
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
env = gym.make('LunarLander-v3', render_mode='rgb_array')

In [ ]:
env.reset()
frame = env.render()

In [ ]:
plt.imshow(frame)
plt.tight_layout()
plt.savefig('LunarLander.png', dpi=300)

In [ ]:
# Define the policy network
class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, 128)
        self.fc2 = nn.Linear(128, action_size)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.softmax(self.fc2(x), dim=1)
        return x

In [ ]:
# Define a function to select an action based on the policy network's output
def select_action(policy_net, state, device=device):
    state = torch.FloatTensor(state).unsqueeze(0).to(device)
    probs = policy_net(state)
    m = Categorical(probs)
    action = m.sample()
    return action.item(), m.log_prob(action)

In [ ]:
# Define the REINFORCE algorithm
def reinforce(env, policy_net, optimizer, num_episodes, device=device, gamma=0.99, max_steps=500):
    total_returns = list()
    for episode in trange(num_episodes):
        state, _ = env.reset()
        rewards = list()
        log_probs = list()
        done = False
        steps = 0
        while not done and steps < max_steps:
            action, log_prob = select_action(policy_net, state)
            next_state, reward, done, _, _ = env.step(action)
            rewards.append(reward)
            log_probs.append(log_prob)
            state = next_state
            steps += 1
        
        R = 0
        policy_loss = list()
        returns = list()
        
        for r in rewards[::-1]:
            R = r + gamma * R
            returns.insert(0, R)
        
        returns = torch.tensor(returns).to(device)
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)
        
        for log_prob, R in zip(log_probs, returns):
            policy_loss.append(-log_prob * R)
        
        optimizer.zero_grad()
        policy_loss = torch.cat(policy_loss).sum()
        policy_loss.backward()
        optimizer.step()
        total_returns.append(R.cpu().detach().item())
        if episode % 100 == 0:
            print(f'Episode {episode}\tLast reward: {sum(rewards)}')

    return total_returns

In [ ]:
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

In [ ]:
policy_net = PolicyNetwork(state_size, action_size).to(device)
optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)

In [ ]:
# Train the policy network using REINFORCE
num_episodes = 5000
total_returns = reinforce(env, policy_net, optimizer, num_episodes)

# Close the environment
env.close()

In [ ]:
def run_episode(env, policy_net, max_steps=500):
    frames = []
    state, _ = env.reset()
    done = False
    steps = 0
    while not done and steps < max_steps:
        frame = env.render()
        frames.append(frame)
        action, _ = select_action(policy_net, state)
        next_state, _, done, _, _ = env.step(action)
        state = next_state
        steps += 1
    
    env.close()
    return frames

# Function to display the captured frames as an animation
def display_frames_as_gif(frames):
    fig = plt.figure()
    plt.axis('off')
    im = plt.imshow(frames[0])

    def update_fig(frame):
        im.set_array(frame)
        return im,

    ani = FuncAnimation(fig, update_fig, frames=frames, interval=50, blit=True)
    ani.save("lunar_lander.gif", writer=PillowWriter(fps=20))
    plt.close(fig)

In [ ]:
# Run an episode and capture frames
frames = run_episode(env, policy_net)

# Display the captured frames as an animation
display_frames_as_gif(frames)

display(Image(filename="lunar_lander.gif"))

In [ ]:
def plot_frames(frames, n, m, a, b, filename):
    selected_frames = frames[::m][:n]
    fig, axes = plt.subplots(a, b, figsize=(20, 20))
    for i in range(a):
        for j in range(b):
            idx = i * b + j
            if idx < m:
                axes[i, j].imshow(selected_frames[idx])
                axes[i, j].axis('off')
            else:
                axes[i, j].axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
# Define the number of frames and the spacing
n = 9  # Number of frames to display
a = 3
b = 3
m = 15  # Frames apart

plot_frames(frames, n, m, a, b, 'lunar_lander.png')

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(total_returns,  'k-', label='Rewards')
plt.xlabel('Time Steps')
plt.ylabel('Reward')
plt.legend()
plt.grid(True)
plt.savefig('REINFORCE_rewards.png', dpi=300)
plt.show()